## Zone Splitting Planning

In [ ]:
from static_bilevel import StaticBilevelModel, FruitInfo
import numpy as np
import time

# Example usage
height = 1.2
length = 1.2
arms_n = 2
vx_max = 0.5
vy_max = 0.5
vz_max = 0.5
ax_max = 1.0
ay_max = 1.0
az_max = 1.0
dt = 0.1
safety_distance = 0.0
seed_num = 100
infeasibility_count = 0
for seed in range(seed_num):
    # Generate random fruits for testing
    rng = np.random.default_rng(seed)
    fruit_n = 8
    depths = [0.2, 0.5]
    X = rng.uniform(0.0, length, fruit_n) # length
    # X = np.array([0.4, 0.5, 0.6, 0.7])
    Y = rng.uniform(depths[0], depths[1], fruit_n)  # depth
    Z = rng.uniform(safety_distance, height - safety_distance, fruit_n)  # height
    # Z = np.array([0.5, 0.7, 0.5, 0.7])
    detach_times = rng.uniform(2.0, 3.0, fruit_n)  # detach times, it could be a function of ripening levels or it could be a constant.
    fruits = [FruitInfo(X[i], Y[i], Z[i], detach_times[i]) for i in range(fruit_n)]
    model = StaticBilevelModel(fruits, height, length, arms_n, vx_max, vy_max, vz_max, ax_max, ay_max, az_max, dt, safety_distance)
    # model.plot_initial_fruits_and_arms()
    tic = time.time()
    try:
        # model.bilevel_trajectory_planning() # Not working for now
        model.zone_splitting_planning(gap=0.05) # Larger gap for faster solving
    except Exception as e:
        print(f"Optimization failed for seed {seed} with error: {e}")
        infeasibility_count += 1
        continue
    toc = time.time()
    print(f"Time taken for optimization: {toc - tic:.3f} seconds")
    print("Optimization completed.")
    # model.draw_trajectories(video=True, show_trajectory=False)
    print(f"Model results are:\n{model.arms_plan}")
    print(f"Trajectories are stored in model.arms_plan.paths_x, paths_y, paths_z.")
print(f"Number of infeasible optimizations: {infeasibility_count} out of {seed_num}")

## Test Trajectory Plan On Sequences

In [ ]:
from static_bilevel import StaticBilevelModel, FruitInfo, ArmsPlan
import numpy as np
import time

# Default parameters for testing
height = 1.2
length = 1.2
vx_max = 0.5
vy_max = 0.5
vz_max = 0.5
ax_max = 1.0
ay_max = 1.0
az_max = 1.0
dt = 0.1

# You can change these parameters to test different scenarios
arms_n = 2
safety_distance = 0.1
fruit_n = 8
seed = 2

# Generate a random fruit distribution
rng = np.random.default_rng(seed)
depths = [0.2, 0.5]
X = rng.uniform(0.0, length, fruit_n)  # length
Y = rng.uniform(depths[0], depths[1], fruit_n)  # depth
Z = rng.uniform(safety_distance, height - safety_distance, fruit_n)  # height
detach_times = rng.uniform(2.0, 3.0, fruit_n)  # detach times, it could be a function of ripening levels or it could be a constant.
fruits = [FruitInfo(X[i], Y[i], Z[i], detach_times[i]) for i in range(fruit_n)]
model = StaticBilevelModel(fruits, height, length, arms_n, vx_max, vy_max, vz_max, ax_max, ay_max, az_max, dt, safety_distance)
# model.fruits (and the `fruits` list above, sorted in place) are now ordered by ascending x.

# Randomly assign the x-sorted fruits to the arms, keeping each arm's share roughly equal
base_count, remainder = divmod(fruit_n, arms_n)
arm_counts = [base_count + 1 if r < remainder else base_count for r in range(arms_n)]
rng.shuffle(arm_counts)  # randomize which arms get an extra fruit
arm_labels = np.repeat(np.arange(arms_n), arm_counts)
rng.shuffle(arm_labels)
sequences = [[] for _ in range(arms_n)]
for fruit_idx, arm in enumerate(arm_labels):
    sequences[arm].append(fruit_idx)  # fruit_idx is already in x-sorted order, so each arm's list stays sorted
print(f"Generated random harvesting sequences: {sequences}")

# Build an initial plan with a heuristic schedule, then test feasibility with trajectory_plan_on_sequences
finishing_time, arriving_times = model.heuristic_schedule_on_sequences(sequences)
arms_plan = ArmsPlan(harvesting_sequences=sequences, arriving_times=arriving_times, finishing_time=finishing_time)

tic = time.time()
is_feasible = model.trajectory_plan_on_sequences(arms_plan, alpha=0.9, use_makespan_as_constraint=False, log_to_console=True, gap=0.05)
toc = time.time()
print(f"Time taken for feasibility check: {toc - tic:.3f} seconds")

if is_feasible:
    model.arms_plan = arms_plan
    print(f"Sequence is FEASIBLE. Model results are:\n{model.arms_plan}")
    # Keep the plotting code ready for later use, but don't call it for now.
    model.draw_trajectories(video=False, show_trajectory=True, save_fig=True, fig_name="trajectories.png")              # static trajectory plot
    model.draw_trajectories(video=True, video_name="trajectories.mp4")      # animated trajectory video
else:
    print("Sequence is INFEASIBLE.")


In [ ]:
print(f"Heuristic finishing time is: {finishing_time:.3f} seconds")
print(f"Heuristic arriving times are: {arriving_times}")
print(f"After trajectory planning, finishing time is: {model.arms_plan.finishing_time:.3f} seconds")
print(f"arriving_times after trajectory planning are: {model.arms_plan.arriving_times} seconds")